# CUDA 真的存在吗：Driver、Runtime 与 Toolkit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rootsongjc/gpu-lab-notebooks/blob/main/labs/zh/lab-03-cuda-exists.ipynb)

用三行命令验证 CUDA 是否可用。搞清楚 NVIDIA Driver、CUDA Runtime、CUDA Toolkit（nvcc）三者的区别，这是后面所有 CUDA 实验和 HAMi 资源管理的基础概念。

本 notebook 配合教程章节 [CUDA 真的存在吗：Driver、Runtime 与 Toolkit](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-03-cuda-exists/) 使用。运行前请确认 **Runtime → Change runtime type → T4 GPU**。

每个 cell 对应教程中一个 Step；Markdown 单元摘录了必要的解释，完整上下文请回到教程阅读。

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap --format=csv

## Why this problem matters

[Lab 01](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-01-meet-your-gpu/) 留了一个问题：`nvidia-smi` 里那个 `CUDA Version: 12.x` 到底是什么意思？很多人据此以为“机器上装好了 CUDA 12.x”，结果跑程序报错。本 Lab 把 NVIDIA 软件栈拆成三层，让你用命令验证每一层，建立后面所有 CUDA 实验（[Lab 07 编程模型](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-07-cuda-programming-model/)、[Lab 08 Kernel](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-08-cuda-kernel/)）和 HAMi 理解的基础。

## CUDA 三层模型

![CUDA 三层模型：程序调用用户态 CUDA Runtime，Runtime 调用内核态 NVIDIA Driver，Driver 驱动 GPU 硬件；nvcc / Toolkit 只在编译期参与](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-03-cuda-exists/cuda-three-layers.svg)

| 层 | 是什么 | 谁提供 | 在 Colab 里 |
| :--- | :--- | :--- | :--- |
| **Driver** | 内核态驱动，管理设备、显存、上下文 | Google 镜像 / 宿主 | 已装（`nvidia-smi` 里的 `Driver Version`） |
| **Runtime** | 用户态库 `libcudart`，程序调用它的 API | 随 CUDA Toolkit 或随框架 wheel 包安装 | 有两份：PyTorch 自带一份，`/usr/local/cuda` 还有一份 |
| **Toolkit** | `nvcc` 编译器、头文件、工具（`cuobjdump`/`compute-sanitizer`） | NVIDIA CUDA Toolkit 安装包 | **已装**（`/usr/local/cuda/bin/nvcc`） |

关键洞察：**`nvidia-smi` 里的 `CUDA Version` 是“驱动能支持的最高 Runtime 版本上限”，不代表 Runtime/Toolkit 已经装到这个版本，也不代表只有一份 Runtime。**

## Step 1：驱动层，GPU 能否被访问

In [ ]:
import torch
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
print("device name:", torch.cuda.get_device_name(0))

预期：

```text
cuda available: True
device count: 1
device name: Tesla T4
```

如果 `cuda available: False`，几乎可以肯定是 Runtime 还在 CPU（见 [Lab 01 Failure Modes](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-01-meet-your-gpu/#failure-modes)）。

## Step 2：Runtime 层，PyTorch 带的 CUDA 版本

In [ ]:
print("torch version:", torch.__version__)
print("bundled CUDA:", torch.version.cuda)

预期类似（具体版本号随 Colab 镜像变化）：

```text
torch version: 2.x.x+cu12y
bundled CUDA: 12.y
```

注意：`torch.version.cuda` 是 **PyTorch wheel 包自己捆绑的 CUDA Runtime 版本**，它和 `nvidia-smi` 的 `CUDA Version` 可以不同：只要 Runtime 版本 ≤ 驱动支持的上限即可（向后兼容）。

> 这就是为什么 `nvidia-smi` 显示 `CUDA Version: 12.4`，而 `torch.version.cuda` 可能是 `12.1` 或 `12.6`，两者不冲突。如果 PyTorch 捆绑的版本**高于**驱动上限，才会出问题（但 CUDA 11.x 以来的“minor version compatibility”让同一大版本内的小版本跨越也能工作）。

## Step 3：Toolkit 层，编译器是否存在

In [ ]:
!which nvcc && nvcc --version | tail -2
!ls /usr/local/cuda/bin | head -20

预期：

```text
/usr/local/cuda/bin/nvcc
Cuda compilation tools, release 12.x, V12.x.xxx
Build cuda_12.x.r12.x/compiler.xxxxxxxx_0
```

Colab 的 GPU runtime **预装了完整的 CUDA Toolkit**，`nvcc` 就在 `/usr/local/cuda/bin`。这是 Colab 作为“CUDA 实验室”最大的便利：你不需要 `apt-get` 任何东西就能编译 `.cu` 文件。[Lab 07](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-07-cuda-programming-model/) 会用它编译第一个 CUDA C++ 程序。

再比对一下三个版本号：

In [ ]:
!nvidia-smi --query-gpu=driver_version --format=csv,noheader
!nvcc --version | grep release
import torch; print(torch.version.cuda)

你很可能会看到 **三个不同的数字**：驱动支持的上限、系统 Toolkit 的版本、PyTorch 捆绑的版本。它们互不冲突，只要后两者都不高于第一个。

> **为什么这对 Kubernetes 很重要**
>
> 在 Kubernetes 上，这三层分别由不同角色负责：**Driver** 装在节点上（GPU Operator 的 driver DaemonSet），**Runtime / Toolkit** 打在容器镜像里（`nvidia/cuda:12.x-devel` 或 PyTorch 镜像）。容器里的 Runtime 版本必须 ≤ 节点驱动支持的上限，否则 Pod 启动后会报 `CUDA driver version is insufficient for CUDA runtime version`。你在 [L1](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/single-gpu/) 装 GPU Operator 时会再遇到这个约束。

## Verification

完成后你应该能回答：

- 驱动版号在哪看？（`nvidia-smi` 的 `Driver Version`）
- Runtime 版本在哪看？（`torch.version.cuda`，或 `nvcc --version` 看系统 Toolkit）
- 为什么 `nvidia-smi` 的 `CUDA Version` 可能 ≠ 实际 Runtime 版本？（它是“驱动支持的上限”）
- 同一台机器上为什么可以同时存在多个 Runtime 版本？（系统 Toolkit 和框架 wheel 各带一份）

## Failure Modes

| 症状 | 原因 | 处理 |
| :--- | :--- | :--- |
| `torch.cuda.is_available()` 返回 `False` | Runtime 是 CPU | 切到 T4 GPU runtime |
| `False` 但 runtime 是 T4 | 你自己 `pip install` 覆盖成了 CPU-only 构建 | `pip install torch --index-url https://download.pytorch.org/whl/cu126`（版本按驱动上限选） |
| `nvcc: command not found` | 极少见；镜像异常或 `PATH` 被改 | `!/usr/local/cuda/bin/nvcc --version`；仍不行则 Disconnect and delete runtime |
| `CUDA driver version is insufficient for CUDA runtime version` | 框架捆绑的 Runtime 高于驱动上限 | 降低框架的 CUDA 版本，或升级驱动（Colab 里做不到后者） |
| `torch.version.cuda` 与 `nvidia-smi` 不一致 | 向后兼容，正常 | 不用处理 |

## 记录你的 Metadata Card

运行完成后，把实际数值回填到教程该章的 Metadata Card：运行日期、驱动 / CUDA 版本、关键输出数值、遇到的 Failure Mode。欢迎在 [rootsongjc/gpu-lab-notebooks](https://github.com/rootsongjc/gpu-lab-notebooks) 提交 PR 或 Issue。